# Udacity free-trial readiness screener

**Business question:** Should Udacity launch a readiness screen before the free trial?

**Answer:** **Do not launch yet.** The screen reduces free-trial enrollments, but the experiment does not rule out meaningful harm to paid conversion.


## 1. Decision framework

- **Success metric — Gross conversion:** reduce trial enrollments by at least 1 percentage point.
- **Guardrail — Net conversion:** paid conversion must not fall by more than 0.75 percentage points.
- **Diagnostic — Retention:** payments per enrollment; useful for interpretation, not a launch criterion.
- Enrollment and payment outcomes are available for only the first 23 days because they require a 14-day maturity window.


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB = ROOT / 'data' / 'processed' / 'analysis.db'
POWERBI = ROOT / 'data' / 'powerbi'


## 2. Data quality and experiment health

In [2]:
daily = pd.read_csv(ROOT / 'data' / 'processed' / 'experiment_daily.csv')
quality = pd.Series({
    'Rows': len(daily),
    'Days per group': daily.groupby('experiment_group').size().min(),
    'Duplicate group-dates': daily.duplicated(['experiment_group', 'event_date']).sum(),
    'Mature days per group': daily.groupby('experiment_group')['payments'].count().min(),
})
print(quality.to_string())


Rows                     74
Days per group           37
Duplicate group-dates     0
Mature days per group    23


In [3]:
sanity = pd.read_csv(POWERBI / 'sanity_checks.csv')
print(sanity.to_string(index=False, formatters={
    'observed': '{:.3%}'.format, 'ci_low': '{:.3%}'.format, 'ci_high': '{:.3%}'.format
}))


                        metric observed  ci_low ci_high status
Pageview allocation to control  50.064% 49.882% 50.118%   Pass
   Click allocation to control  50.047% 49.588% 50.412%   Pass
 Click-through-rate difference   0.006% -0.124%  0.135%   Pass


All pre-treatment checks pass: traffic is balanced and the click-through-rate difference is compatible with random variation.

## 3. SQL model and mature funnel

In [4]:
query = '''
SELECT experiment_group, mature_pageviews, mature_clicks, enrollments, payments,
       gross_conversion, net_conversion, retention
FROM group_summary
ORDER BY experiment_group;
'''
with sqlite3.connect(DB) as connection:
    summary = pd.read_sql_query(query, connection)
print(summary.to_string(index=False, formatters={
    'gross_conversion': '{:.2%}'.format,
    'net_conversion': '{:.2%}'.format,
    'retention': '{:.2%}'.format,
}))


experiment_group  mature_pageviews  mature_clicks  enrollments  payments gross_conversion net_conversion retention
         Control            212163          17293       3785.0    2033.0           21.89%         11.76%    53.71%
      Experiment            211362          17260       3423.0    1945.0           19.83%         11.27%    56.82%


The screener acts between the trial click and enrollment. Therefore the main comparison uses mature clicks as the denominator for both trial and paid conversion.

## 4. Metric effects and uncertainty

In [5]:
results = pd.read_csv(POWERBI / 'metric_results.csv')
print(results.to_string(index=False, formatters={
    'control_rate': '{:.2%}'.format, 'experiment_rate': '{:.2%}'.format,
    'difference': '{:+.2%}'.format, 'ci_low': '{:+.2%}'.format,
    'ci_high': '{:+.2%}'.format, 'practical_threshold': '{:+.2%}'.format,
}))


          metric control_rate experiment_rate difference ci_low ci_high practical_threshold            role                 status
Gross conversion       21.89%          19.83%     -2.06% -2.91%  -1.20%              -1.00%  Success metric                   Pass
  Net conversion       11.76%          11.27%     -0.49% -1.16%  +0.19%              -0.75%       Guardrail                   Fail
       Retention       53.71%          56.82%     +3.11% +0.81%  +5.41%              +1.00% Diagnostic only Not a launch criterion


## 5. Decision

1. Gross conversion fell by **2.06 pp**; its confidence interval is entirely beyond the **−1.00 pp** practical target. The screen successfully filters trial enrollments.
2. Net conversion fell by **0.49 pp**, but its 95% CI is **−1.16 to +0.19 pp**. Because the lower bound crosses the **−0.75 pp** guardrail, meaningful harm to paid conversion cannot be excluded.
3. Only **34,553 of 54,826** required mature clicks were observed (about **63%** of the planned information).

**Recommendation:** do not launch the screen to all users. Continue the experiment to the planned sample size or test a less restrictive message, then launch only if the paid-conversion guardrail passes.
